# Trial-wise responses analysis - CLMM for acceptance

- [x] Acceptance: cumulative link mixed model
- [x] authorship
---
stats: 
1. to avoid type I error, False Discovery Rate (FDR) correction
2. effect size: odds ratios (OR) with 95% confidence intervals (CI95%) for pairwise contrasts
---
notes for codes
- For a CLMM, mode = "latent" computes estimated marginal means on the model’s underlying latent log-odds scale, not directly on the observed 1–10 rating scale. contrast estimate = difference in log cumulative odds
- `pairs(emm, adjust = "holm")` - controls the family-wise error rate across a family of pairwise tests. It is generally less conservative than Bonferroni. 
- [?] FDR correction -> `pairs(emm, adjust = "BH")` `pairs(emm, adjust = "fdr")`

In [57]:
library(utils)
library(ordinal)
library(emmeans)
library(dplyr)
library(tidyverse)   # read_csv, dplyr, ggplot2, etc.
library(DHARMa)

In [58]:
# Single ordinal response analysed using CLMM
li_resp_CLMM <- c("Accept", "Authorship")

wide <- read_csv(
  "data_analysis/trial_wise_SoA_with_pp_traits.csv",
  show_col_types = FALSE
)
head(wide, 3)

pp,trial,voice_id,condition,scene_id,1,2,3,4,5,⋯,8,9,ans_sd,SoPA,SoNA,Accept,SoA,Authorship,AI_literacy_z,DoC_z
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
P01,1,clone,repeat,1,4,6,1,4,2,⋯,2,3,1.752549,2.8,4.5,3,3.000000,4,1.370731,-0.06454751
P01,2,clone,enhance,3,2,2,3,2,1,⋯,3,7,1.832251,2.0,5.5,7,2.142857,2,1.370731,-0.06454751
P01,3,clone,counter,5,1,2,2,1,2,⋯,3,4,2.375470,1.6,6.0,4,1.714286,1,1.370731,-0.06454751


In [59]:
wide$Accept <- ordered(wide$Accept)  # ordered
wide$Authorship <- ordered(wide$Authorship)  # ordered

wide$condition <- factor(
    wide$condition,
    levels=c("repeat","enhance","counter")
)
head(wide, 3)

pp,trial,voice_id,condition,scene_id,1,2,3,4,5,⋯,8,9,ans_sd,SoPA,SoNA,Accept,SoA,Authorship,AI_literacy_z,DoC_z
<chr>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<ord>,<dbl>,<ord>,<dbl>,<dbl>
P01,1,clone,repeat,1,4,6,1,4,2,⋯,2,3,1.752549,2.8,4.5,3,3.000000,4,1.370731,-0.06454751
P01,2,clone,enhance,3,2,2,3,2,1,⋯,3,7,1.832251,2.0,5.5,7,2.142857,2,1.370731,-0.06454751
P01,3,clone,counter,5,1,2,2,1,2,⋯,3,4,2.375470,1.6,6.0,4,1.714286,1,1.370731,-0.06454751


# CLMM - acceptance
SoA ~ condition * voice_id + AI_literacy_z + DoC_z + (1 | pp)

In [60]:
# 2 model fitting
model_accept <- clmm(
    Accept ~
        voice_id * condition +
        AI_literacy_z +
        DoC_z +
        (1|pp),
    data=wide,
)

# 3 model summary
summary(model_accept)

# 4 joint tests
joint_tests(model_accept)

# 5 post hoc pairwise comparisons
emm <- emmeans(model_accept,
            ~ condition,
            mode="latent")

cell_contrasts <- pairs(
    emm,
    adjust = "BH"
)

# 6 effect size (odds ratio) and 95% CI
cell_OR <- summary(
    cell_contrasts,
    infer = c(TRUE, TRUE)
) %>%
    as.data.frame() %>%
    mutate(
        OR = exp(estimate),
        CI95_low = exp(asymp.LCL),
        CI95_high = exp(asymp.UCL)
    ) %>%
    select(
        contrast,
        OR,
        CI95_low,
        CI95_high,
        z.ratio,
        p.value
    )

cell_OR

Cumulative Link Mixed Model fitted with the Laplace approximation

formula: Accept ~ voice_id * condition + AI_literacy_z + DoC_z + (1 |      pp)
data:    wide

 link  threshold nobs logLik  AIC     niter      max.grad cond.H 
 logit flexible  326  -662.74 1359.48 2012(6039) 1.25e-03 4.2e+02

Random effects:
 Groups Name        Variance Std.Dev.
 pp     (Intercept) 0.6625   0.8139  
Number of groups:  pp 28 

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
voice_idrobotic                   0.54332    0.35285   1.540   0.1236    
conditionenhance                 -0.67905    0.33060  -2.054   0.0400 *  
conditioncounter                 -2.56938    0.36243  -7.089 1.35e-12 ***
AI_literacy_z                     0.17605    0.19270   0.914   0.3609    
DoC_z                            -0.08559    0.19421  -0.441   0.6594    
voice_idrobotic:conditionenhance  0.05252    0.48237   0.109   0.9133    
voice_idrobotic:conditioncounter -0.85281    0.49952  -

,model term,df1,df2,F.ratio,Chisq,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,voice_id,1,Inf,1.914,1.914,1.664837e-01
3,condition,2,Inf,56.240,112.480,3.762025e-25
4,AI_literacy_z,1,Inf,0.835,0.835,3.609367e-01
5,DoC_z,1,Inf,0.194,0.194,6.594463e-01
2,voice_id:condition,2,Inf,2.115,4.230,1.205892e-01


NOTE: Results may be misleading due to involvement in interactions



,contrast,OR,CI95_low,CI95_high,z.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,repeat - enhance,1.920884,1.070736,3.446034,2.673942,7.496547e-03
2,repeat - counter,20.001042,9.937091,40.257425,10.252672,3.453506e-24
3,enhance - counter,10.412417,5.408047,20.047615,8.562048,1.663203e-17


In [61]:
names(summary(cell_contrasts, infer = TRUE))

[1] "contrast"  "estimate"  "SE"        "df"        "asymp.LCL" "asymp.UCL"
[7] "z.ratio"   "p.value"

# CLMM - authorship

In [63]:
# 2
model_author <- clmm(
    Accept ~
        voice_id * condition +
        AI_literacy_z +
        DoC_z +
        (1|pp),
    data=wide,
)

# 3
summary(model_author)

# 4
joint_tests(model_author)

# 5
emm <- emmeans(model_author,
            ~ condition,
            mode="latent")

cell_contrasts <- pairs(
    emm,
    adjust = "BH"
)

# 6 effect size (odds ratio) and 95% CI
cell_OR <- summary(
    cell_contrasts,
    infer = c(TRUE, TRUE)
) %>%
    as.data.frame() %>%
    mutate(
        OR = exp(estimate),
        CI95_low = exp(asymp.LCL),
        CI95_high = exp(asymp.UCL)
    ) %>%
    select(
        contrast,
        OR,
        CI95_low,
        CI95_high,
        z.ratio,
        p.value
    )

cell_OR

Cumulative Link Mixed Model fitted with the Laplace approximation

formula: Accept ~ voice_id * condition + AI_literacy_z + DoC_z + (1 |      pp)
data:    wide

 link  threshold nobs logLik  AIC     niter      max.grad cond.H 
 logit flexible  326  -662.74 1359.48 2012(6039) 1.25e-03 4.2e+02

Random effects:
 Groups Name        Variance Std.Dev.
 pp     (Intercept) 0.6625   0.8139  
Number of groups:  pp 28 

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
voice_idrobotic                   0.54332    0.35285   1.540   0.1236    
conditionenhance                 -0.67905    0.33060  -2.054   0.0400 *  
conditioncounter                 -2.56938    0.36243  -7.089 1.35e-12 ***
AI_literacy_z                     0.17605    0.19270   0.914   0.3609    
DoC_z                            -0.08559    0.19421  -0.441   0.6594    
voice_idrobotic:conditionenhance  0.05252    0.48237   0.109   0.9133    
voice_idrobotic:conditioncounter -0.85281    0.49952  -

,model term,df1,df2,F.ratio,Chisq,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,voice_id,1,Inf,1.914,1.914,1.664837e-01
3,condition,2,Inf,56.240,112.480,3.762025e-25
4,AI_literacy_z,1,Inf,0.835,0.835,3.609367e-01
5,DoC_z,1,Inf,0.194,0.194,6.594463e-01
2,voice_id:condition,2,Inf,2.115,4.230,1.205892e-01


NOTE: Results may be misleading due to involvement in interactions



,contrast,OR,CI95_low,CI95_high,z.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,repeat - enhance,1.920884,1.070736,3.446034,2.673942,7.496547e-03
2,repeat - counter,20.001042,9.937091,40.257425,10.252672,3.453506e-24
3,enhance - counter,10.412417,5.408047,20.047615,8.562048,1.663203e-17
